# 📗 학습 전후 모델 비교: 도메인 평가, 일반 능력 퇴행과 퇴행 줄이기

교안 01이 에폭마다 저장한 체크포인트를 학습 전 모델(Base)과 같은 지표로 평가해, 규칙으로 Tuned 하나를 고릅니다. 이어서 일반 능력 퇴행을 점검하고, 가중치 보간으로 퇴행을 줄여 봅니다. 지표는 MLflow에 기록하고, 결정 테이블로 서비스에 쓸 모델을 고릅니다.

<img src="images/10_lesson02_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 평가로 모델을 고르고 일반 능력 퇴행 줄이기'이고 아래 띠는 '학습 데이터 · 노무 평가셋 · 범위 밖 질문 · 일반 질문'입니다. 네 판이 1에서 4로 이어집니다. 1 '평가 지표'(학습 데이터 정답률 · 도메인 정답률 · 거절률)는 '학습 데이터'와 '노무 질문'이 '판사'를 거쳐 '정답률'로, '범위 밖 질문'이 '거절 키워드'를 거쳐 '거절률'로 가는 두 줄, 2 '에폭별 평가와 모델 고르기'(학습 데이터 80% 이상 -> 도메인 정답률 최고)는 '1에폭'부터 '4에폭'까지의 체크포인트 카드가 '1단계: 학습 데이터 80% 이상', '2단계: 도메인 정답률 최고' 깔때기를 지나 'Tuned' 하나로 가고, 'Base' 카드는 깔때기 밖에 '비교 기준'으로 따로 놓인 그림, 3 '일반 능력 퇴행'(일반 질문 정답률 · perplexity)은 같은 '일반 질문'을 Base와 Tuned에 나란히 묻는 그림, 4 '퇴행 줄이기와 결정'(α = 0.5, 0.75 보간)은 회색 원과 주황 원을 섞은 두 원, 회색과 주황이 반씩인 'α = 0.5' 원과 주황이 더 큰 'α = 0.75' 원이 각각 '결정 테이블'로 가는 그림입니다.">

**학습 목표** (그림의 번호와 같습니다)

1. **정답률은 Jev 판사로, 거절률은 거절 키워드로 계산합니다.**
2. **에폭별 체크포인트를 같은 지표로 평가하고 규칙으로 Tuned를 고릅니다.**
3. **일반 질문 정답률과 perplexity로 일반 능력 퇴행을 점검합니다.**
4. **가중치 보간으로 퇴행을 줄여 보고, 지표를 MLflow에 기록해 결정 테이블로 쓸 모델을 고릅니다.**

## 1. 실습 준비

### 1-1. 라이브러리와 경로

교안 01과 같은 준비 셀입니다. 체크포인트와 MLflow 기록도 같은 `output/` 폴더에서 읽습니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 1-2. 학습 전 모델

Base를 불러옵니다. 체크포인트(`output/labor_epochs4_lr5e-6`)는 평가할 때 하나씩 불러옵니다. **교안 01을 끝까지 실행한 뒤 시작합니다.**

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저는 학습 전후 모델이 같습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 학습 전 모델(Base)은 허브에서 불러옵니다(교안 01과 같은 float32)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
print("Base:", MODEL_NAME)

### 1-3. 교안 01과 같은 함수

`load_jsonl`은 앞 시간과 같습니다. 평가 문항이 많아 생성은 질문 32개를 한 번에 생성하는 `generate_replies`로 바꿉니다(패딩 때문에 한두 문항의 답이 조금 달라질 수 있습니다).

In [ ]:
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문 목록을 32개씩 묶어 한 번에 생성하고, 질문 순서대로 새로 만든 답 목록을 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    # 여러 질문을 한 번에 생성하려면 짧은 질문 앞을 패딩으로 채웁니다. 답은 오른쪽 끝에 이어 쓰므로 패딩을 왼쪽에 둡니다
    tokenizer.padding_side = "left"

    answers = []
    for start in range(0, len(questions), 32):
        # 1) 질문마다 학습 데이터와 같은 chat template으로 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
                enable_thinking=False,       # 빈 생각 블록을 붙여 추론 과정 없이 바로 답합니다
            ))
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,  # 길이가 다른 질문을 패딩으로 맞춰 한 텐서로 묶습니다
        ).to(model.device)

        # 2) 대화 끝 토큰까지 또는 max_new_tokens 까지 답을 만듭니다
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한
                do_sample=False,                       # 그리디 생성: 같은 질문에 같은 답
                eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
                pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
            )

        # 3) 입력 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤에 붙은 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    tokenizer.padding_side = "right"
    return answers

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

기록은 교안 01과 같은 MLflow 실험에 남깁니다.

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("labor-sft")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

### 1-4. Jev 판사

답이 맞았는지는 Jev 판사로 판정합니다. LLM 판사 단원에서 G-Eval과 Jev를 모두 배웠고, 이 교안은 그 단원 부록처럼 DeepEval `JevEval`을 씁니다. Jev는 글을 쓰지 않는 판정 모델로, 판정 문장(`Noul`)을 주면 그 문장이 참일 확률을 돌려줍니다.

- **API 키:** `.env`의 `TYPESAFE_API_KEY`가 필요합니다. 모델 하나를 평가할 때마다 문항 수만큼 Jev를 부릅니다.
- **통과:** `strict_mode=True`라 판정 문장이 모두 참일 확률 0.5 이상일 때만 통과입니다. 점수는 통과면 1, 아니면 0입니다.
- **경계에 선 문항:** 참일 확률이 0.5 근처인 문장은 작은 차이로 통과와 실패가 갈리므로, 한두 문항 차이는 조심해서 읽습니다. 그래도 고르는 규칙은 통과 수가 하나라도 많은 쪽을 고릅니다.

In [ ]:
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

채점 기준은 두 가지입니다. 일반 질문과 학습 데이터 표본은 정답 답변만 보고, 노무·전월세 질문은 근거 조문(`context`)도 함께 봅니다. 정답 답변에 없지만 조문과 맞는 사실을 답했을 때 지어낸 내용으로 보지 않게 하려는 것입니다.

판정 문장에는 테스트 케이스 필드 이름(`input`, `actual_output`, `expected_output`, `context`)을 그대로 쓰고, 한 문장에 한 가지만 묻습니다. 거절 답 행은 학습 데이터 표본에서 이미 뺐고 노무 평가셋에는 거절을 정답으로 둔 문항이 없어, 거절을 맞는 답으로 보는 문장은 두지 않습니다.

In [ ]:
# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    # 1. 질문에 직접 답했는가
    Noul("actual_output은 input의 질문에 직접 답한다."),
    # 2. 사실과 결론이 정답 답변과 같은가(표현이 달라도 같은 정보면 같다고 봅니다)
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    # 3. 질문과 관계없는 지어낸 내용을 덧붙이지 않았는가
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    # 4. 같은 말을 되풀이하지 않았는가
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 노무, 전월세 질문용: 근거 조문(context)도 함께 봅니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        # 5. 정답 답변에 없는 말은 근거 조문과 어긋나지 않는가
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


# 판사를 동시에 부르는 설정(LLM 판사 단원과 같은 방법, 동시에 보내는 상한은 이 교안 64)
judge_config = {
    # 동시에 보낼 문항 수 상한. 반 전체가 같은 Jev 키(TYPESAFE_API_KEY)를 쓰면 429(요청 한도 초과)가 날 수 있어, 429가 나면 값을 줄입니다
    "async_config": AsyncConfig(max_concurrent=64),
    "display_config": DisplayConfig(print_results=False, show_indicator=False),  # 긴 기본 출력을 끕니다
}


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다.

    행에 근거 조문(law_text)이 있으면 근거 조문 기준으로, 없으면 일반 기준으로 채점합니다.
    """
    law_cases = []
    general_cases = []
    for index in range(len(rows)):
        row = rows[index]
        if "law_text" in row:
            law_cases.append(LLMTestCase(
                name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
                input=row["question"],                    # 질문
                actual_output=answers[index],             # 채점할 모델의 답
                expected_output=row["reference_answer"],  # 정답 답변
                context=[row["law_text"]],                # 근거 조문 원문
            ))
        else:
            general_cases.append(LLMTestCase(
                name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
                input=row["question"],                    # 질문
                actual_output=answers[index],             # 채점할 모델의 답
                expected_output=row["reference_answer"],  # 정답 답변
            ))

    judged = {}
    for test_cases, metric in [(law_cases, law_metric), (general_cases, general_metric)]:
        if len(test_cases) == 0:
            continue
        result = evaluate(test_cases=test_cases, metrics=[metric], **judge_config)
        for test_result in result.test_results:
            metric_data = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
            # 점수는 모든 문장이 참이면 1, 아니면 0입니다. 이유에는 문장마다 참일 확률 P(yes)가 나옵니다
            judged[test_result.name] = (metric_data.score, metric_data.success, metric_data.reason)

    judged_list = []
    for index in range(len(rows)):
        judged_list.append(judged[str(index)])
    return pd.DataFrame(judged_list, columns=["점수", "통과", "이유"])

## 2. 평가 지표: Base 먼저

이 절과 다음 절은 지표를 하나씩 정의하고 Base로 계산합니다. 체크포인트도 같은 함수로 계산합니다. 아래 지표 일곱 개를 모든 모델에 똑같이 계산해 한 테이블로 비교합니다. 판사 통과는 판정 문장이 모두 참일 확률 0.5 이상일 때입니다.

| 지표 | 무엇을 보나 | 계산 | 좋은 방향 |
|---|---|---|---|
| 학습 데이터 정답률 | 학습한 질문에 학습한 답을 내는지(배웠는가) | 학습 데이터 표본 질문 가운데 판사 통과 비율 | 클수록 좋음. 1에 가까워야 학습이 된 것 |
| 도메인 정답률(재현) | 같은 상황을 다르게 물어도 배운 답을 꺼내는지 | 평가셋 재현 문항 가운데 판사 통과 비율 | 클수록 좋음 |
| 도메인 정답률(적용) | 배운 조문을 학습 질문에 없던 구체적 조건(날짜, 인원 등)에 맞춰 쓰는지 | 평가셋 적용 문항 가운데 판사 통과 비율 | 클수록 좋음 |
| 과잉 거절률 | 답해야 할 노무 질문까지 거절하는지 | 평가셋 질문 가운데 거절 문장으로 답한 비율 | 작을수록 좋음(0이 목표) |
| 거절률 | 범위 밖 질문을 정해진 문장으로 거절하는지 | 범위 밖 질문 가운데 거절 문장으로 답한 비율 | 클수록 좋음 |
| 일반 질문 정답률 | 노무와 관계없는 상식 질문에 맞게 답하는지(일반 능력) | 시스템 프롬프트 없이 묻고 판사 통과 비율 | 클수록 좋음. Base보다 떨어지면 퇴행 |
| 일반 perplexity | 노무와 관계없는 일반 글을 얼마나 잘 예측하는지(일반 능력) | 일반 문단 토큰의 평균 손실에 exp | 작을수록 좋음. Base보다 커지면 퇴행 |

### 2-1. 평가셋과 학습 데이터 표본

검증 손실은 생성한 답이 맞는지 알려 주지 않으므로, 질문마다 정답 답변(`reference_answer`)을 붙인 평가셋으로 생성한 답을 채점합니다.

- **학습 데이터 정답률:** 학습한 질문 40개(거절 답 행 제외)를 다시 물어 학습한 답을 내는지 봅니다. 학습한 답도 못 내는 모델은 새 질문도 못 맞히므로 맨 먼저 봅니다.
- **재현 문항:** 학습 데이터에 같은 상황이 있고 표현만 다른 문항입니다. 배운 답을 다시 꺼내는지 봅니다.
- **적용 문항:** 근거 조문은 학습 데이터가 다룬 것이지만, 질문 속 구체적 조건(날짜, 인원, 금액 같은 값)은 학습 질문에 없는 문항입니다. 답하려면 배운 조문을 그 조건에 맞춰 계산하거나 판단해야 합니다(예: 해고된 날과 오늘 날짜를 주고 구제신청 기한이 지났는지 묻기). 두 정답률을 나눠 봐야 오른 이유가 외움인지 적용인지 가를 수 있습니다.
- **평가셋 문항의 근거:** 재현과 적용 모두 학습 데이터가 다룬 조문만 근거로 합니다. 배운 적 없는 조문을 묻지 않으므로, 틀린 답은 모르는 사실이 아니라 배운 것을 꺼내거나 적용하지 못한 결과입니다.

같은 조문(근로기준법 제23조, 해고 제한)의 실제 문항으로 보면 이렇게 다릅니다.

| 구분 | 질문 | 맞히려면 |
|---|---|---|
| 학습 데이터 | "회사에서 특별한 이유도 설명하지 않고 저를 해고하거나 휴직·정직·전직·감봉시키려고 하는데, 이렇게 해도 되나요?" | (학습한 답) 정당한 이유 없이 해고, 전직, 감봉 등을 할 수 없습니다 |
| 재현 문항 | "징계성 인사발령으로 갑자기 지방 발령이 났어요. 정당한 이유가 없어도 회사가 이렇게 할 수 있나요?" | 표현만 바뀐 같은 사실을 다시 꺼내면 됩니다(정당한 이유 없는 전직은 안 됨) |
| 적용 문항 | "직원 20명인 회사에서 출산휴가를 마치고 복귀한 지 40일이 지났는데 해고 통보를 받았어요. 출산휴가 뒤에는 해고할 수 없다던데 이 해고는 무조건 무효인가요?" | 배운 조문의 기간(휴가 기간과 그 뒤 30일)을 질문의 조건(복귀 40일)에 대어 판단해야 합니다(무조건 무효는 아니고, 정당한 이유는 여전히 필요)

지표는 아래 다섯 파일(`data/본실습/`)로 계산합니다.


<img src="images/11_eval_matrix.png" width="1050" alt="제목은 '평가 데이터 다섯 가지를 서로 다른 조건과 채점으로 비교합니다'입니다. 열은 평가 데이터, 시스템 프롬프트, 채점, 좋은 결과입니다. '학습 데이터 질문'은 노무 상담 프롬프트, '판사가 정답 답변과 비교', '학습한 답을 맞힘'. '노무 평가셋(재현·적용)'은 노무 상담 프롬프트, '판사가 정답 답변과 비교', 'Tuned 정답률이 오름'. '범위 밖 질문'은 노무 상담 프롬프트, '거절 문장 키워드', '거절률은 오름'과 '과잉 거절은 낮게'. '일반 질문'은 '프롬프트 없음', '판사가 정답 답변과 비교', 'Base만큼 유지'. '일반 문단'은 '프롬프트 없음', 'perplexity 계산', 'Base만큼 유지'. 맨 아래 띠는 '정답은 판사가, 고정된 거절 문장만 키워드로 셉니다'입니다.">


| 파일 | 한 행의 내용 | 계산하는 지표 |
|---|---|---|
| `labor_train.jsonl` (거절 답 행을 빼고 고르게 뽑은 40행) | 교안 01이 학습한 노무 질문과 답(`answer`) | 학습 데이터 정답률 |
| `labor_eval.jsonl` (교안 01에서 쓰지 않고 남겨 둔 평가셋) | 노무 질문, 정답 답변, 문항 종류(`question_type`: 재현·적용), 근거 조문(`doc_id`와 원문 `law_text`) | 도메인 정답률(재현, 적용), 과잉 거절률 |
| `out_of_scope_eval.jsonl` | 범위 밖 질문, 거절 키워드(`keywords`), 질문 종류(`kind`) | 거절률 |
| `general_questions.jsonl` | 상식 질문, 정답 답변 | 일반 질문 정답률 |
| `general_texts.json` | 노무와 관계없는 일상 문단 하나 | 일반 문단 perplexity |

- **오염 검사:** 평가 질문은 학습 질문과 글자 3-gram 겹침이 0.7 미만, 일반 문단은 학습 글 전체와 0.5 미만입니다. 뜻만 같은 질문은 사람이 읽어 재현으로 분류했습니다.

In [ ]:
# 노무 평가셋: question_type 은 재현, 적용, law_text 는 판사에게 넘길 근거 조문 원문입니다
eval_rows = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")
eval_table = pd.DataFrame(eval_rows)
print("평가 문항 수:", len(eval_rows), "| 종류별:", eval_table["question_type"].value_counts().to_dict())

# 종류마다 앞 세 문항을 봅니다
display(eval_table.groupby("question_type").head(3)[["question_type", "question", "reference_answer", "doc_id"]])

In [ ]:
# 교안 01이 학습한 노무 학습 데이터
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")

# 거절 답 행은 뺍니다(거절은 범위 밖 질문으로 따로 계산합니다)
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)

# 학습 데이터 표본: 남은 행 수를 40으로 나눈 간격마다 한 행씩, 앞에서 끝까지 고르게 뽑습니다
sample_gap = len(answer_rows) // 40
train_sample_rows = []
for row in answer_rows[::sample_gap][:40]:
    # 학습한 답(answer)을 정답 답변으로 씁니다. 근거 조문 원문(law_text)이 없어 판사는 일반 기준으로 채점합니다
    train_sample_rows.append({"question": row["question"], "reference_answer": row["answer"]})
print("학습 행 수:", len(train_rows), "| 거절 답을 뺀 행 수:", len(answer_rows), "| 뽑는 간격:", sample_gap, "| 표본 수:", len(train_sample_rows))

### 2-2. 답 생성과 판사 채점

맞는 숫자를 다른 조건에 붙이면 틀린 답이고, 낱말이 달라도 맞는 답일 수 있어 키워드로는 정답을 판정할 수 없습니다. 그래서 판사가 정답 답변과 뜻을 비교합니다. 모델마다 같은 방법으로 생성하고 채점하도록 `answer_and_judge` 하나로 묶고, 판사 호출은 `evaluate`로 동시에 보냅니다.

In [ ]:
def has_keywords(answer, keywords):
    """키워드가 답에 모두 들어 있으면 True를 돌려줍니다."""
    for keyword in keywords:
        if keyword not in answer:
            return False
    return True


def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 한꺼번에 생성하고 판사가 동시에 채점해, 질문마다 한 행씩 모읍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": has_keywords(answer, ["답할 수 없"])})  # 아래 거절률과 과잉 거절률에 씁니다
    result_table = pd.DataFrame(result_rows)

    # 범위 밖 질문은 정답 답변이 없어 판사를 부르지 않습니다
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table


# 노무 상담 시스템 프롬프트로 Base의 답을 만들고 채점합니다. 판사 호출 수 = 표본 수 + 평가 문항 수
base_train_table = answer_and_judge(base_model, train_sample_rows, SYSTEM_PROMPT)
base_domain_table = answer_and_judge(base_model, eval_rows, SYSTEM_PROMPT)

# 노무 평가셋의 앞 다섯 문항: 답, 판정, 판정 이유
display(base_domain_table[["종류", "질문", "답변", "통과", "이유"]].head())

### 2-3. 학습 데이터 정답률과 도메인 정답률

판정 문장이 모두 참일 확률 0.5 이상이면 통과로 보고, 통과한 질문의 비율을 학습 데이터 표본과 노무 평가셋(재현, 적용 따로)에서 계산합니다.

In [ ]:
# 통과 비율은 True를 1, False를 0으로 본 평균입니다. 통과 수 / 문항 수로도 봅니다
print(f"학습 데이터 정답률: {base_train_table['통과'].mean():.3f} ({base_train_table['통과'].sum()}/{len(base_train_table)})")
for question_type in ["재현", "적용"]:
    base_part = base_domain_table[base_domain_table["종류"] == question_type]
    print(f"도메인 정답률({question_type}): {base_part['통과'].mean():.3f} ({base_part['통과'].sum()}/{len(base_part)})")

**결과 읽기**
- **Base 학습 데이터 5/40, 재현 3/29, 적용 1/20:** 강조와 목록을 섞어 길게 답하고 숫자를 지어 냅니다(배우자 출산휴가를 20일이 아니라 10일로 답함).

### 2-4. 범위 밖 질문 거절과 과잉 거절

학습 데이터는 범위 밖 질문에 고정된 거절 문장을 가르쳤으므로, 거절 여부는 판사 없이 거절 키워드 `답할 수 없`가 답에 있는지로 셉니다.

- **거절률:** 학습한 다섯 법으로 답할 수 없는 범위 밖 질문 가운데 거절한 비율입니다. 높을수록 좋습니다.
- **과잉 거절률:** 답할 수 있는 노무 평가셋 질문 가운데 거절한 비율입니다. 낮을수록 좋습니다.

In [ ]:
# 범위 밖 질문: 질문, 거절 키워드, 질문 종류
out_of_scope_rows = load_jsonl(data_dir / "본실습" / "out_of_scope_eval.jsonl")
print("범위 밖 질문 수:", len(out_of_scope_rows))

# 서비스와 같은 노무 상담 시스템 프롬프트로 묻습니다(정답 답변이 없어 판사는 부르지 않습니다)
base_refusal_table = answer_and_judge(base_model, out_of_scope_rows, SYSTEM_PROMPT)
display(base_refusal_table[["질문", "답변", "거절"]])

# 거절률은 범위 밖 질문의 거절 비율, 과잉 거절률은 앞에서 만든 노무 평가셋 답의 거절 비율입니다
print("거절률:", round(base_refusal_table["거절"].mean(), 3), "| 과잉 거절률:", round(base_domain_table["거절"].mean(), 3))

**결과 읽기**
- **Base 거절률 0/12, 과잉 거절률 0/49:** 거절 문장을 배우기 전이라 범위 밖 질문 12개에 모두 답을 지어 냅니다.

## 3. 일반 능력과 Tuned 고르기

**일반 능력 퇴행(regression)**
- 도메인 데이터로 학습한 뒤, 학습 전 모델이 하던 일반 질문 답하기나 일반 글 예측이 나빠지는 것입니다. 심하면 파국적 망각(catastrophic forgetting)이라고 부릅니다.
- 도메인 지표에는 드러나지 않으므로, 일반 질문과 일반 문단으로 Base와 비교합니다.


<img src="images/12_regression.png" width="1050" alt="제목은 '도메인 정답률이 오르는 동안 일반 능력이 떨어질 수 있습니다'입니다. 왼쪽 '도메인 정답률' 막대는 Base보다 Tuned가 높고 '올라감', 오른쪽 '일반 능력' 막대는 Tuned가 Base보다 낮고 '떨어짐: 일반 능력 퇴행'이 적혀 있습니다. 맨 아래 띠는 '좁은 데이터로 가중치 전체를 크게 바꿀수록 원래 하던 일을 잊기 쉽습니다'입니다.">


### 3-1. 일반 질문

노무 시스템 프롬프트 아래에서는 상식 질문도 거절이 맞는 답이므로, 시스템 프롬프트 없이(`system_prompt=None`) 묻고 같은 판사로 채점합니다. 클수록 좋고, Base보다 떨어지면 일반 능력이 퇴행한 것입니다.

In [ ]:
# 일반 질문: 질문과 정답 답변
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")
print("일반 질문 수:", len(general_rows))

# 시스템 프롬프트 없이 묻고 채점합니다. 판사 호출 수 = 일반 질문 수
base_general_table = answer_and_judge(base_model, general_rows, None)
display(base_general_table[["질문", "답변", "통과"]])

# 일반 질문 정답률은 통과 열의 평균입니다
print("일반 질문 정답률:", round(base_general_table["통과"].mean(), 3))

### 3-2. 일반 문단 perplexity

일반 질문 정답률은 한 문항이 값을 크게 바꾸므로, 답을 생성하지 않고 일반 문단의 토큰 수천 개를 얼마나 잘 예측하는지도 봅니다.

**perplexity(PPL)**
- 모델이 다음 토큰을 고를 때 평균 몇 개의 후보 사이에서 헷갈리는지를 나타낸 값입니다. 작을수록 그 글을 잘 예측합니다.
- 정답 토큰마다 확률 1/4을 줬다면 PPL은 4(후보 넷 가운데 찍는 만큼)이고, 늘 확률 1이면 가장 작은 값 1입니다.


<img src="images/15_perplexity.png" width="1050" alt="제목은 'perplexity: 다음 토큰을 고를 때 헷갈리는 후보 수'입니다. 파랑 띠는 일반 문단 앞부분 '비가 그치지 않아 빨래가' 뒤에 다음 토큰 자리 '?'가 있습니다. 왼쪽 판은 정답 토큰에 확률 1을 준 모델로 후보 카드 하나만 있고 'PPL 1: 헷갈리지 않음', 오른쪽 판은 정답 토큰에 확률 1/4을 준 모델로 같은 크기의 후보 카드 네 장이 있고 'PPL 4: 후보 4개 가운데 찍는 만큼 헷갈림'입니다. 맨 아래 띠는 '학습 뒤 PPL이 Base보다 커지면 일반 글을 덜 잘 예측합니다'입니다.">



계산은 교안 01의 손실(교차 엔트로피)을 평균 낸 뒤 exp를 하는 것입니다.

$\displaystyle \text{PPL} = \exp\Big(\frac{1}{N}\sum_{i=1}^{N} -\log p(x_i \mid x_{<i})\Big)$

- **$x_i$:** 문단의 $i$번째 토큰 **하나**이고, $x_{<i}$는 그 앞의 토큰 **전부**입니다. $N$은 예측한 토큰의 개수입니다(문단 첫 토큰은 뺌).
- **exp를 하는 이유:** 확률 1/4의 손실 $\log 4 \approx 1.39$를 exp로 다시 4로 돌려, '헷갈리는 후보 수'로 읽게 합니다.

In [ ]:
# 일반 문단: 노무와 관계없는 일상 문단입니다. 학습에 쓰는 글 전체와 글자 3-gram 겹침이 0.5 미만인 문단만 골랐습니다
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))
print("일반 문단 수:", len(general_texts), "| 전체 글자 수:", len("".join(general_texts)))
display(general_texts[:3])

In [ ]:
import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 교차 엔트로피 평균에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

        with torch.no_grad():
            loss = model(
                input_ids,
                labels=input_ids,  # 입력을 정답으로 주면 다음 토큰을 하나씩 예측한 손실의 평균을 돌려줍니다
            ).loss.item()

        # 첫 토큰은 앞 토큰이 없어 예측하지 않으므로 예측한 토큰 수는 길이 - 1. 평균 * 개수로 이 문장의 손실 합을 만듭니다
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted

    # 전체 손실 합 / 전체 예측 토큰 수 = 평균 손실, 여기에 exp
    return math.exp(total_loss / total_tokens)


# Base의 일반 문단 perplexity
base_general_ppl = perplexity(base_model, general_texts)
print("Base 일반 perplexity:", round(base_general_ppl, 1))

### 3-3. 지표 일곱 개 모으기

도메인 지표와 일반 능력을 함께 판단하도록 지표 일곱 개를 모델마다 한 행에 모읍니다. `evaluate_model`이 앞의 평가를 모두 되풀이하고 `table_metrics`가 지표를 계산합니다. Base부터 `model_metrics`에 모으고, 뒤에 나오는 모델도 여기에 행을 더합니다.

In [ ]:
# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나에 앞의 평가를 모두 되풀이해 채점 테이블 네 개와 perplexity 를 사전으로 돌려줍니다.

    판사 호출 수 = 표본 수 + 평가 문항 수 + 일반 질문 수
    """
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }


# Base는 앞에서 만든 채점 테이블을 같은 모양의 사전으로 모읍니다(답을 다시 생성하지 않습니다)
base_tables = {"train": base_train_table, "domain": base_domain_table, "refusal": base_refusal_table,
               "general": base_general_table, "ppl": base_general_ppl}
model_metrics = {"Base": table_metrics(base_tables)}

### 3-4. 에폭별 체크포인트 평가

교안 01이 lr 5e-6로 저장한 4개 체크포인트를 하나씩 불러 같은 지표를 계산하고 지웁니다. 에폭이 늘수록 학습 데이터와 재현은 오르는데 적용과 일반 능력이 떨어지면 학습 데이터를 외우는 중입니다.

**Tuned를 고르는 규칙** (결과를 보기 전에 정해 둡니다)

평가한 체크포인트 가운데 비교 기준이 될 하나를 아래 규칙으로 고르고, 이 모델을 Tuned라고 부릅니다. 결과를 본 뒤에 규칙을 정하면 그 결과에 맞춘 규칙이 되므로, 평가하기 전에 적어 둡니다.

1. **학습 데이터 정답률 0.8 이상(40문항 가운데 32문항)만 후보로 둡니다.** 학습이 제대로 됐다면 이미 본 답은 대부분 맞혀야 합니다.
2. **후보 가운데 도메인 통과 수(재현 + 적용)가 가장 많은 것을 고릅니다.** 같으면 가중치를 덜 바꾼 앞 에폭을 고릅니다.
3. **후보가 없으면 학습 데이터 정답률이 가장 높은 것을 고릅니다.**

In [ ]:
import gc

# 교안 01이 에폭마다 저장한 체크포인트(폴더 이름 checkpoint-<갱신 횟수>)
epoch_dir = output_dir / "labor_epochs4_lr5e-6"
checkpoint_steps = []
for path in epoch_dir.glob("checkpoint-*"):
    checkpoint_steps.append(int(path.name.split("-")[1]))
checkpoint_steps.sort()
print("체크포인트 갱신 횟수:", checkpoint_steps)

# 체크포인트마다 모델을 불러 채점 테이블과 지표 일곱 개를 계산하고, 다 쓴 모델은 지워 GPU 메모리를 비웁니다
epoch_tables = {}
epoch_metrics = {}
epoch_labels = {}
for epoch, step in enumerate(checkpoint_steps, start=1):
    checkpoint_model = AutoModelForCausalLM.from_pretrained(epoch_dir / f"checkpoint-{step}", dtype=torch.float32).to(device)
    epoch_tables[step] = evaluate_model(checkpoint_model)
    epoch_metrics[step] = table_metrics(epoch_tables[step])
    epoch_labels[step] = f"{epoch}에폭(step {step})"
    model_metrics[epoch_labels[step]] = epoch_metrics[step]
    del checkpoint_model
    gc.collect()
    torch.cuda.empty_cache()

forgetting_table = pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)
display(forgetting_table.round(3))

**결과 읽기**
- **학습 데이터 14/40 -> 39/40, 재현 3/29 -> 10/29, 적용 0~1/20:** 학습한 질문은 거의 다 맞히지만 적용은 오르지 않아, 학습 데이터를 외우는 쪽으로 갑니다.
- **일반 perplexity 49.96 -> 64.70(Base 36.43):** 일반 질문 정답률은 Base 근처인데 perplexity는 에폭마다 커져, 퇴행이 perplexity에 먼저 드러납니다.

### 3-5. Tuned 고르기

평가 전에 정해 둔 세 규칙을 그대로 코드로 옮깁니다. 규칙 3이 없으면 후보가 하나도 없을 때 `tuned_step`이 None으로 남습니다. 고른 체크포인트는 이름표 앞에 `Tuned`를 붙여, 뒤의 모든 테이블에서 같은 이름으로 보이게 합니다.

In [ ]:
# 학습 데이터 정답률이 이 값 이상인 체크포인트만 Tuned 후보로 둡니다
TRAIN_ACC_MIN = 0.8

# 체크포인트마다 도메인 통과 수(재현 + 적용)를 셉니다. 노무 평가셋 문항은 모두 재현이나 적용입니다
domain_passes = {}
selection_rows = []
for step in checkpoint_steps:
    domain_passes[step] = int(epoch_tables[step]["domain"]["통과"].sum())
    selection_rows.append({"체크포인트": epoch_labels[step], "학습 데이터 정답률": epoch_metrics[step]["train_acc"],
                           "후보": epoch_metrics[step]["train_acc"] >= TRAIN_ACC_MIN, "도메인 통과 수": domain_passes[step]})
display(pd.DataFrame(selection_rows).round(3))

# 규칙 1, 2: 후보 가운데 도메인 통과 수가 가장 많은 체크포인트. 앞 에폭부터 보며 더 많을 때만 바꾸므로 같으면 앞 에폭이 남습니다
tuned_step = None
for step in checkpoint_steps:
    if epoch_metrics[step]["train_acc"] >= TRAIN_ACC_MIN:
        if tuned_step is None or domain_passes[step] > domain_passes[tuned_step]:
            tuned_step = step

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 체크포인트(같으면 앞 에폭)
if tuned_step is None:
    tuned_step = checkpoint_steps[0]
    for step in checkpoint_steps:
        if epoch_metrics[step]["train_acc"] > epoch_metrics[tuned_step]["train_acc"]:
            tuned_step = step

# 고른 체크포인트의 에폭 번호(checkpoint_steps 에서의 자리 + 1)
tuned_epoch = checkpoint_steps.index(tuned_step) + 1

# 고른 체크포인트의 이름표 앞에 Tuned 를 붙이고, model_metrics 의 에폭 행을 새 이름표로 다시 담습니다(행 순서는 그대로)
epoch_labels[tuned_step] = f"Tuned {tuned_epoch}에폭(step {tuned_step})"
model_metrics = {"Base": model_metrics["Base"]}
for step in checkpoint_steps:
    model_metrics[epoch_labels[step]] = epoch_metrics[step]
print("고른 체크포인트:", epoch_labels[tuned_step], "| 에폭 수:", tuned_epoch, "| 폴더:", epoch_dir / f"checkpoint-{tuned_step}")

**결과 읽기**
- **Tuned 4에폭(step 1172):** 후보 3에폭(36/40)과 4에폭(39/40) 가운데 도메인 통과 수가 많은 4에폭(10문항, 3에폭 7문항)입니다.
- **조금 낙관적인 값:** 고른 평가셋으로 비교도 하므로, 실무에서는 마지막 확인용 테스트셋을 따로 둡니다.

### 3-6. Base와 Tuned 비교

Tuned의 답을 Base와 나란히 봅니다. 답과 판정은 앞에서 남긴 `epoch_tables`에서 꺼냅니다.

In [ ]:
# 고른 Tuned의 채점 테이블
tuned_tables = epoch_tables[tuned_step]
tuned_domain_table = tuned_tables["domain"]

# 노무 평가셋: 같은 질문의 두 답과 판정을 나란히 봅니다
domain_compare_table = pd.DataFrame({
    "종류": base_domain_table["종류"], "질문": base_domain_table["질문"],
    "Base 답변": base_domain_table["답변"], "Base 통과": base_domain_table["통과"],
    "Tuned 답변": tuned_domain_table["답변"], "Tuned 통과": tuned_domain_table["통과"],
    "Tuned 판정 이유": tuned_domain_table["이유"],
})
display(domain_compare_table)

# 종류별 통과 수 / 문항 수. 한 문항이 정답률을 1 / 문항 수 만큼 바꾸고 경계에 선 문항은 판정이 쉽게 갈리므로, 한두 문항 차이는 조심해서 해석합니다
for question_type in ["재현", "적용"]:
    base_part = base_domain_table[base_domain_table["종류"] == question_type]
    tuned_part = tuned_domain_table[tuned_domain_table["종류"] == question_type]
    print(f"{question_type}: Base {base_part['통과'].sum()}/{len(base_part)}, Tuned {tuned_part['통과'].sum()}/{len(tuned_part)}")

**결과 읽기**
- **재현 3/29 -> 10/29, 적용 1/20 -> 0/20:** 재현은 올랐지만, 배운 조문을 학습 질문에 없던 조건에 맞춰 쓰는 적용 문항은 오르지 않았습니다.
- **틀린 적용 예:** 유산 25일째 질문에 Tuned는 "25일 이내에 청구해야 하므로 청구할 수 있습니다"라고 답했습니다. 조문은 20일 이내라, 말투는 맞고 숫자를 바꿔 결론이 틀렸습니다.

In [ ]:
# 범위 밖 질문: 같은 질문의 두 답과 거절 여부
tuned_refusal_table = tuned_tables["refusal"]
display(pd.DataFrame({"질문": base_refusal_table["질문"], "Base 답변": base_refusal_table["답변"], "Base 거절": base_refusal_table["거절"],
                      "Tuned 답변": tuned_refusal_table["답변"], "Tuned 거절": tuned_refusal_table["거절"]}))

In [ ]:
# 일반 질문: 시스템 프롬프트 없이 물은 두 답과 판정
tuned_general_table = tuned_tables["general"]
display(pd.DataFrame({"질문": base_general_table["질문"], "Base 답변": base_general_table["답변"], "Base 통과": base_general_table["통과"],
                      "Tuned 답변": tuned_general_table["답변"], "Tuned 통과": tuned_general_table["통과"]}))

**결과 읽기**
- **거절률 0/12 -> 4/12, 과잉 거절률 0/49 그대로:** 노무 질문은 하나도 거절하지 않았지만, 범위 밖 8문항은 여전히 노무 조문 말투로 지어 답합니다.
- **일반 질문:** 거절한 답은 없고 답이 짧은 한 문장으로 바뀌었지만, "일주일은 며칠인가요?"에 "1주 = 52시간"처럼 근로시간 숫자가 섞인 답이 있습니다.

## 4. 퇴행 줄이기 방법

Tuned는 일반 질문 정답률은 지켰지만(5/30 -> 7/30) 일반 perplexity가 36.43에서 64.70으로 커졌습니다. 더 오래 학습하면 퇴행이 커질 수 있습니다. 줄이는 방법 넷 가운데 학습률·학습량 줄이기는 교안 01에서 해 봤고, 이 교안에서는 다시 학습하지 않는 가중치 보간을 실습합니다.


<img src="images/13_forgetting_fixes.png" width="1050" alt="제목은 '일반 능력 퇴행을 줄이는 방법'입니다. 카드 네 장: 1 '학습률·학습량 줄이기'(가중치를 덜 바꿈, 교안 01), 2 '일반 데이터 섞기'(일반 질문과 답을 함께 학습, 오늘은 실습하지 않음), 3 '가중치 보간'(Base와 마지막 체크포인트를 섞음, 회색 원과 주황 원을 섞은 두 원으로 반반인 'α = 0.5'와 주황이 더 큰 'α = 0.75', 이 교안 실습), 4 '일부만 학습'(LoRA, 다음 시간). 맨 아래 띠는 '도메인 정답률과 일반 능력을 함께 비교해 고릅니다'입니다.">


| 기법 | 하는 일 | 드는 비용과 한계 | 이 교안 |
|---|---|---|---|
| 학습률·학습량 줄이기 | 학습률이나 에폭을 줄여 가중치를 덜 바꿉니다 | 도메인 정답률도 덜 오를 수 있습니다 | 교안 01 학습률 비교, 앞의 에폭별 체크포인트 |
| 일반 데이터 섞기(replay) | 도메인 학습 데이터에 일반 질문과 답을 섞어 함께 학습합니다 | 섞을 답을 만드는 생성 시간, 섞은 만큼 늘어나는 학습 시간 | 오늘은 사용하지 않습니다(성능도 측정하지 않습니다) |
| 가중치 보간(WiSE-FT) | Base와 학습한 모델의 가중치를 비율 α로 섞어 새 모델을 만듭니다 | 다시 학습하지 않음, α 후보마다 평가, 학습한 행동도 함께 줄어듦 | 아래에서 실습 |
| 일부만 학습(LoRA) | 원래 가중치는 그대로 두고 더한 작은 가중치만 학습합니다 | 퇴행이 적은 만큼 도메인을 배우는 양도 줄어듭니다 | 다음 시간 |

**일반 데이터 섞기를 쓸 때 주의할 점**
- 섞을 답은 학습 전 모델이 쓴 답이라도 틀리거나 같은 문장을 되풀이하는 답이 섞여 있어, 그대로 섞으면 그 답까지 배웁니다. 섞기 전에 답을 걸러야 합니다.
- 학습률과 에폭을 이미 작게 골라 퇴행이 작다면 섞어서 얻는 이득이 작고, 섞은 만큼 학습 시간만 늡니다.

## 5. 가중치 보간

**가중치 보간(weight interpolation)**
- 같은 구조의 두 모델에서 이름이 같은 가중치끼리 비율 α로 섞어 새 모델을 만드는 방법입니다. 학습 전후 모델을 섞는 이 방법을 [WiSE-FT](https://arxiv.org/abs/2109.01903)라고 부릅니다.
- **목적:** 다시 학습하지 않고, 학습한 능력은 되도록 남기면서 일반 능력을 되찾습니다.
- **섞을 두 모델:** Base와, 가장 오래 학습해 학습 데이터에 가장 많이 맞춰진 마지막(4에폭) 체크포인트입니다.

가중치 하나 $\theta$마다 다음과 같이 섞습니다. $\theta_{\text{last}}$는 마지막 체크포인트의 가중치입니다.

$\displaystyle \theta_\alpha = (1-\alpha)\,\theta_{\text{Base}} + \alpha\,\theta_{\text{last}}$

- **α:** 마지막 체크포인트의 비율입니다. α = 0이면 Base, α = 1이면 마지막 체크포인트입니다. 예: 0.2010과 0.2030을 α = 0.5로 섞으면 0.2020입니다.
- **α 두 개(0.5, 0.75):** 두 값을 같은 지표로 비교해 맞바꿈을 봅니다. α가 클수록 학습한 것(학습 데이터 답, 거절, 말투)을 더 남기는 대신 일반 능력은 덜 되찾습니다. 실무에서는 후보를 더 촘촘히 두고 거절률까지 함께 보고 고릅니다.

### 5-1. 두 모델의 가중치

두 모델의 가중치를 `state_dict()`(이름에서 텐서로 가는 사전)로 꺼냅니다.

In [ ]:
# 마지막 에폭 체크포인트(갱신 횟수가 가장 큰 폴더)를 불러옵니다
last_step = checkpoint_steps[-1]
last_model = AutoModelForCausalLM.from_pretrained(epoch_dir / f"checkpoint-{last_step}", dtype=torch.float32).to(device)
print("섞을 체크포인트:", epoch_labels[last_step])

# 두 모델의 가중치 사전: 가중치 이름 -> 텐서
base_state = base_model.state_dict()
last_state = last_model.state_dict()

# 섞을 마지막 체크포인트의 비율 α 두 개(클수록 마지막 체크포인트에 가깝습니다)
interp_alphas = [0.5, 0.75]

### 5-2. 보간 모델의 지표

모델은 하나만 만들고, α마다 섞은 가중치를 `load_state_dict`로 넣어 같은 지표 일곱 개를 계산합니다.

In [ ]:
# 섞은 가중치를 넣을 모델을 만듭니다
interp_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# α마다 이름이 같은 텐서끼리 (1 - α) * Base + α * 마지막 체크포인트로 섞어 넣고 지표 일곱 개를 계산합니다
interp_labels = []
for interp_alpha in interp_alphas:
    mixed_state = {}
    for name in base_state:
        mixed_state[name] = (1 - interp_alpha) * base_state[name] + interp_alpha * last_state[name]
    interp_model.load_state_dict(mixed_state)

    # 보간 모델의 이름표와 지표 일곱 개
    interp_label = f"보간 α={interp_alpha}"
    model_metrics[interp_label] = table_metrics(evaluate_model(interp_model))
    interp_labels.append(interp_label)

# 마지막 체크포인트와 섞은 가중치는 다 썼으므로 지워 GPU 메모리를 비웁니다(base_state 와 interp_model 은 따라하기에서 다시 씁니다)
del last_model, last_state, mixed_state
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# model_metrics 에서 Base(α = 0), 보간 모델, 마지막 체크포인트(α = 1)를 α 순서로 골라 봅니다
# 마지막 체크포인트가 Tuned 이면 그 이름표에 Tuned 가 붙어 있습니다
interp_rows = ["Base"] + interp_labels + [epoch_labels[last_step]]
interp_compare_table = pd.DataFrame(model_metrics).T.loc[interp_rows].rename(columns=METRIC_NAMES)
display(interp_compare_table.round(3))

**결과 읽기**
- **α 0.5: 일반 질문 11/30, perplexity 40.67(Tuned 7/30, 64.70):** perplexity는 Base(36.43) 가까이 돌아왔고, 일반 질문은 Base(5/30)보다 높습니다. 대신 학습 데이터 17/40, 재현 2/29, 거절률 0/12(Tuned 39/40, 10/29, 4/12)로 학습한 능력의 대부분과 거절을 모두 잃었습니다.
- **α 0.75:** [측정 뒤: α 0.75의 지표와 α 0.5·Tuned와의 비교]

## 6. 기록과 결정

### 6-1. MLflow에 평가 지표 기록하기

평가 지표를 MLflow에 기록으로 남깁니다. 학습한 run에 평가 지표를 더하면 설정, 손실, 평가 지표가 run 하나에 모입니다. 모델은 아래 결정 테이블로 고르고, MLflow 기록은 나중에 다시 찾아보는 데 씁니다. 끝난 run은 `mlflow.start_run(run_id=...)`로 다시 엽니다.

- **에폭 run:** 체크포인트마다 지표를 그 갱신 횟수(step)로 남기고, Tuned는 태그(`tuned_step`, `tuned_epoch`)로 표시합니다.
- **Base와 보간 모델:** 학습한 run이 없어 새 run을 만듭니다. 보간 모델은 α마다 run 하나입니다.

교안 01의 명령으로 MLflow 화면을 켜면 기록한 run을 볼 수 있습니다.

In [ ]:
# 같은 실험의 run 목록에서 run 이름으로 run_id 를 찾습니다. 다시 실행해 같은 이름이 여러 개면 가장 최근 것(맨 위)을 씁니다
runs_table = mlflow.search_runs(experiment_names=["labor-sft"], order_by=["start_time DESC"])
epoch_run_id = runs_table[runs_table["tags.mlflow.runName"] == "epochs4_lr_5e-6"]["run_id"].iloc[0]
print("찾은 run:", epoch_run_id)

# 에폭 run 에는 체크포인트의 갱신 횟수를 step 으로 넣어, MLflow 화면에서 학습 기록과 같은 가로축에 놓습니다
with mlflow.start_run(run_id=epoch_run_id):
    for step in epoch_metrics:
        mlflow.log_metrics(epoch_metrics[step], step=step)

    # 고른 Tuned 는 태그로 남깁니다(태그는 다시 실행하면 새 값으로 바뀝니다)
    mlflow.set_tags({"tuned_step": tuned_step, "tuned_epoch": tuned_epoch})

In [ ]:
# Base는 학습한 run 이 없으므로 새 run 을 만듭니다
with mlflow.start_run(run_name="base"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_metrics(model_metrics["Base"])

# 보간 모델은 α마다 새 run 을 만들고, α와 섞은 체크포인트를 남깁니다
for interp_alpha in interp_alphas:
    with mlflow.start_run(run_name=f"interp_{interp_alpha}"):
        mlflow.log_params({"alpha": interp_alpha, "mixed_checkpoint": str(epoch_dir / f"checkpoint-{last_step}")})
        mlflow.log_metrics(model_metrics[f"보간 α={interp_alpha}"])

### 6-2. 서비스에 쓸 모델 고르기

모든 모델의 지표를 한 테이블로 보고 서비스에 쓸 모델을 고릅니다.

In [ ]:
# 모든 모델의 지표 일곱 개. 고른 체크포인트의 이름표에는 앞에서 Tuned 를 붙였습니다
summary_table = pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)
display(summary_table.round(3))

**결정**

| 결정 | 근거 | 다시 검토할 조건 |
|---|---|---|
| 노무 상담 서비스에는 Tuned 4에폭(step 1172)을 씁니다 | 재현 0.345(Base 0.103)로 도메인 통과 수가 가장 많고, 과잉 거절 없이 거절률 0.333입니다. 보간 α 0.5는 거절률 0이라 상담 모델로 쓸 수 없습니다. 보간 α 0.75는 [측정 뒤: α 0.75의 지표와 α 0.5·Tuned와의 비교] | 적용 정답률이 0이므로 조문 사실은 RAG로 찾아 답하게 하고, 범위 밖 질문은 가드레일로 앞에서 거릅니다. 일반 질문 비중이 큰 서비스면 보간 α 0.75와 다음 시간 LoRA를 함께 비교합니다. 평가는 그리디 생성이었으므로, 서비스 생성 설정(샘플링)으로 답을 몇 번 생성해 같은 결론인지 확인합니다. |

### 6-3. 고른 모델 저장

고른 Tuned 체크포인트를 `output/labor_sft_final`에 최종 모델로 저장합니다. 다음 단계(서빙, 다음 시간 비교)는 이 폴더를 쓰므로 학습은 다시 하지 않습니다(디스크 약 3GB).

In [ ]:
# 고른 Tuned 체크포인트를 불러옵니다
final_model = AutoModelForCausalLM.from_pretrained(epoch_dir / f"checkpoint-{tuned_step}", dtype=torch.float32).to(device)

# 평가셋 첫 질문을 물어, 불러온 모델이 고른 Tuned 인지 답으로 확인합니다
question = eval_rows[0]["question"]
print("질문:", question)
print("답변:", generate_replies(final_model, [question], SYSTEM_PROMPT)[0])

모델과 토크나이저를 한 폴더에 저장하고, 저장한 파일을 확인합니다.

In [ ]:
# 모델과 토크나이저를 최종 모델 폴더에 저장합니다
final_dir = output_dir / "labor_sft_final"
final_model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

# 저장된 파일 이름과 크기(MB)
final_saved_files = []
for path in sorted(final_dir.iterdir()):
    final_saved_files.append({"파일": path.name, "크기(MB)": round(path.stat().st_size / 1024 ** 2, 1)})
display(pd.DataFrame(final_saved_files))

# 저장한 모델은 다 썼으므로 지워 GPU 메모리를 비웁니다
del final_model
gc.collect()
torch.cuda.empty_cache()

## 7. 🖐️ 함께 따라하기: 전월세 상담 모델 평가와 퇴행 줄이기

교안 01 따라하기에서 저장한 전월세 상담 모델(`output/lease_lr5e-6`)을 본실습 순서대로 평가하고 가중치 보간을 해 봅니다. 이 폴더가 없으면 교안 01 정답 노트북의 같은 절을 실행합니다.

**본실습과 다른 점**
- **평가셋:** `data/따라하기/lease_eval.jsonl`과 `SYSTEM_PROMPT_LEASE`로 묻고, 정답률 하나로 봅니다.
- **Base 값과 퇴행 줄이기:** Base의 일반 능력은 본실습 값을 그대로 씁니다. 본실습처럼 가중치 보간으로 퇴행을 줄이고, 간단하게 두 α 가운데 α = 0.5 하나만 봅니다.

**실습 진행 방법:** 작성 셀의 `...`만 채웁니다. 본실습의 함수와 변수를 그대로 쓰므로 본실습을 먼저 실행합니다. 완성 코드는 `정답/교안_02_정답.ipynb`의 같은 절에 있습니다.

### 7-1. 모델과 평가셋

In [ ]:
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."

# 교안 01 함께 따라하기에서 저장한 전월세 상담 모델과 평가셋(question, reference_answer, law_text)
lease_model = AutoModelForCausalLM.from_pretrained(output_dir / "lease_lr5e-6", dtype=torch.float32).to(device)
lease_eval_rows = load_jsonl(data_dir / "따라하기" / "lease_eval.jsonl")
print("평가 문항 수:", len(lease_eval_rows))
display(pd.DataFrame(lease_eval_rows)[["question", "reference_answer", "doc_id"]].head())

### 7-2. 판사로 평가셋 채점

In [ ]:
# 🖐️ 함께 따라하기: 전월세 평가셋은 어떤 시스템 프롬프트로 묻나요? answer_and_judge 의 셋째 인자를 채우세요
lease_base_table = answer_and_judge(base_model, lease_eval_rows, ...)
lease_table = answer_and_judge(lease_model, lease_eval_rows, ...)

# 도메인 정답률은 통과 열의 평균입니다
lease_base_acc = lease_base_table["통과"].mean()
lease_acc = lease_table["통과"].mean()

### 7-3. 일반 질문 퇴행 점검

In [ ]:
# 🖐️ 함께 따라하기: 원래 능력을 보는 일반 질문은 어떤 시스템 프롬프트로 묻나요? answer_and_judge 의 셋째 인자를 채우세요
lease_general_acc = answer_and_judge(lease_model, general_rows, ...)["통과"].mean()

# perplexity 는 시스템 프롬프트 없이 문단만 넣습니다
lease_general_ppl = perplexity(lease_model, general_texts)

### 7-4. 가중치 보간 α = 0.5

In [ ]:
lease_state = lease_model.state_dict()

# 🖐️ 함께 따라하기: 이름이 같은 가중치끼리 (1 - α) * Base + α * lease 를 α = 0.5 로 섞으세요
lease_mixed_state = {}
for name in base_state:
    lease_mixed_state[name] = ...

# 섞은 가중치를 본실습의 interp_model 에 넣습니다
interp_model.load_state_dict(lease_mixed_state)

# 보간 모델의 세 지표: 도메인은 전월세 프롬프트, 일반 질문은 시스템 프롬프트 없이
lease_interp_acc = answer_and_judge(interp_model, lease_eval_rows, SYSTEM_PROMPT_LEASE)["통과"].mean()
lease_interp_general_acc = answer_and_judge(interp_model, general_rows, None)["통과"].mean()
lease_interp_general_ppl = perplexity(interp_model, general_texts)

# 세 모델의 세 지표를 한 테이블로 봅니다. Base 의 일반 능력은 같은 조건(시스템 프롬프트 없음)으로 계산한 본실습 값입니다
lease_summary_table = pd.DataFrame([
    {"모델": "Base", "도메인 정답률": lease_base_acc, "일반 질문 정답률": model_metrics["Base"]["general_acc"], "일반 perplexity": base_general_ppl},
    {"모델": "lease", "도메인 정답률": lease_acc, "일반 질문 정답률": lease_general_acc, "일반 perplexity": lease_general_ppl},
    {"모델": "보간 α=0.5 (lease 50%)", "도메인 정답률": lease_interp_acc, "일반 질문 정답률": lease_interp_general_acc, "일반 perplexity": lease_interp_general_ppl},
])
display(lease_summary_table.round(3))

### 7-5. 결과 해석

위 테이블로 세 가지를 판단합니다.
- lease 모델의 도메인 정답률이 Base보다 높은지
- 일반 질문 정답률과 perplexity가 Base보다 나빠졌는지
- 보간 α = 0.5에서 도메인 정답률과 일반 능력이 각각 어떻게 바뀌었는지, 서비스에 어느 모델을 쓸지

**내 판단** (이 셀을 두 번 눌러 적습니다. 모범 해석은 정답 노트북의 같은 자리에 있습니다)

- 도메인 정답률:
- 일반 능력:
- 쓸 모델과 이유:

## 8. 핵심 정리

- **도메인 평가:** 정답은 Jev 판사로, 고정된 거절 문장은 키워드로 셉니다. Base -> Tuned 재현 정답률은 0.103 -> 0.345입니다.
- **일반 능력 퇴행:** 시스템 프롬프트 없는 일반 질문 정답률과 일반 문단 perplexity로 점검합니다. Tuned는 perplexity가 36.43 -> 64.70으로 커졌습니다.
- **Tuned 고르기:** 학습 데이터 정답률 0.8 이상인 체크포인트 가운데 도메인 통과 수가 가장 많은 것을 고릅니다(4에폭).
- **가중치 보간:** Base와 마지막 체크포인트를 $(1-\alpha)\,\theta_{\text{Base}} + \alpha\,\theta_{\text{last}}$로 섞고, α 0.5, 0.75 두 값으로 학습한 것과 일반 능력의 맞바꿈을 봅니다. α 0.5는 일반 능력을 되찾았지만 거절을 모두 잃었고, α 0.75는 [측정 뒤: α 0.75의 지표와 α 0.5·Tuned와의 비교]. 노무 상담에는 Tuned를 씁니다.

다음 시간에는 원래 가중치는 두고 더한 작은 가중치만 학습하는 LoRA를 배웁니다.